# Gradient maps visualization

### Load model and titles

In [78]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import random
from IPython.display import HTML, display
import matplotlib
import matplotlib.cm as cm
import re
import torch
import gc

gc.collect()

torch.cuda.empty_cache()


In [79]:
st_model = SentenceTransformer('BAAI/bge-large-en-v1.5')
base_model = st_model._first_module().auto_model
tokenizer = st_model.tokenizer

DATASET = 'Amazon_Sports_and_Outdoors'

ITEM_PATH = f'../../{DATASET}/{DATASET}.item'

df = pd.read_csv(ITEM_PATH, delimiter='\t')
df = df[['item_id:token', 'title:token']]

titles = df['title:token'].astype('str').tolist()

### Calculate vanilla saliency map

In [80]:
title = random.choice(titles)
inputs = tokenizer(title, return_tensors='pt', add_special_tokens=True)

device = next(base_model.parameters()).device
input_ids = inputs['input_ids'].to(device)
attention_mask = inputs['attention_mask'].to(device)

embeds = base_model.embeddings.word_embeddings(input_ids)
embeds.retain_grad()

output = base_model(inputs_embeds=embeds, attention_mask=attention_mask)
hidden_state = output.last_hidden_state

sentence_embedding = hidden_state.mean(dim=1)

target = sentence_embedding.norm()
target.backward()

token_gradients = embeds.grad.norm(dim=-1).squeeze().detach().cpu().numpy()
tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze())
token_importance = {t: w.item() for t, w in zip(tokens, token_gradients) if t not in tokenizer.all_special_tokens}

In [81]:
def merge_scores(tokens, scores):
    merged_tokens, merged_scores = [], []
    current_token, current_scores = "", []

    for tok, score in zip(tokens, scores):
        if tok.startswith("##"):
            current_token += tok[2:]
            current_scores.append(score)
        else:
            if current_token:
                merged_tokens.append(current_token)
                merged_scores.append(sum(current_scores))
            current_token = tok
            current_scores = [score]
    if current_token:
        merged_tokens.append(current_token)
        merged_scores.append(sum(current_scores))
        
    return merged_tokens, merged_scores

def make_html(tokens, scores):
    norm = matplotlib.colors.Normalize(vmin=min(scores), vmax=max(scores))
    cmap = matplotlib.colormaps.get_cmap("Reds")

    def colorize(token, score):
        rgba = cmap(norm(score))
        rgb = tuple(int(x * 255) for x in rgba[:3])
        return f"<span style='background-color: rgba({rgb[0]}, {rgb[1]}, {rgb[2]}, 0.5); padding:2px 4px; border-radius:4px; margin:1px;'>{token}</span>"

    html = " ".join(colorize(t, s) for t, s in zip(tokens, scores))
    return html

tokens = list(token_importance.keys())
scores = list(token_importance.values())

merged_tokens, merged_scores = merge_scores(tokens, scores)
html = make_html(merged_tokens, merged_scores)
display(HTML(html))